# EGM 722 Project Notebook: Greenspace Analysis (Draft)

## Introduction

Welcome to my project notebook! This will take you through the code required to perform analysis on the amount of Greenspace available in Northern Ireland. 

The code will consist of 4 main parts:
1. **Importing the necessary modules**
2. **Loading the data and setting the CRS**
3. **Carrying out the analysis**
4. **Mapping the results**

**Please Note:** This is currently only a draft - parts may be messy or incomplete!

## Part 1: Preparing Data

To start the project, we first need to import the necessary modules, followed by the data.

In [3]:
#Import the required modules
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from cartopy.feature import ShapelyFeature
import cartopy.crs as ccrs

In [4]:
# Load and name the data layers
greenspace = gpd.read_file(os.path.abspath('data_files/Greenspace_Phase2_06052022.shp'))
lgd = gpd.read_file(os.path.abspath('data_files/LGD.shp'))
dea = gpd.read_file(os.path.abspath('data_files/dea.shp'))
dz = gpd.read_file(os.path.abspath('data_files/DZ2021.shp'))
sdz = gpd.read_file(os.path.abspath('data_files/SDZ2021.shp'))


In [24]:
# View the first 5 rows of the greenspace data to check if it loaded in correctly
dea.head(6)

,OBJECTID,DEA,N_SAs,popn,age18_64,N_inc_dep,N_emp_dep,pc_inc_dep,pc_emp_dep,N_Value,...,R_Criminal,R_Drugs,R_Robbery,R_Shoplift,R_Vehicle_,R_Violence,Shape_Leng,Shape_Area,geometry,Area_SQKM
0,1.0,MACEDON,60,19846,11850,2594,2992,13.070644,25.248945,9525,...,9.523330,3.375995,0.251940,10.682253,1.511640,20.104807,17542.729894,1.005002e+07,"POLYGON ((334156.322 383816.138, 334159.203 38...",10.050021
1,2.0,AIRPORT,42,23143,13513,2312,1673,9.990062,12.380670,8327,...,6.092555,3.197511,0.129629,1.512336,2.549367,17.888778,100844.338308,2.937776e+08,"POLYGON ((326291.638 387137.93, 326300.102 387...",293.777554
2,3.0,ANTRIM,62,23459,14377,2631,2950,11.215312,20.518884,10153,...,14.109723,5.072680,0.255765,4.006991,1.236199,30.180315,24142.561884,2.350514e+07,"POLYGON ((315209.815 389900.74, 315233.431 389...",23.505139
3,4.0,BALLYCLARE,41,18104,10785,1843,1539,10.180071,14.269819,7621,...,4.308440,1.822802,0.165709,1.988511,0.497128,8.561644,53023.566135,7.796712e+07,"POLYGON ((328093.219 395176.251, 328095.159 39...",77.967121
4,5.0,THREE MILE WATER,52,20412,13156,1978,1971,9.690378,14.981757,8162,...,4.703116,2.106604,0.097982,0.293945,0.440917,13.717421,32513.229485,1.879701e+07,"POLYGON ((329144.24 387618.703, 329146.815 387...",18.797012
5,6.0,GLENGORMLEY URBAN,60,19737,11888,1797,1877,9.104727,15.789031,8455,...,5.522622,2.381314,0.101333,1.317323,0.709328,13.325227,16736.203775,9.227570e+06,"POLYGON ((331206.514 384197.638, 331212.837 38...",9.227570


Now that the data is loaded and looks okay, we will check the crs of each layer, and make any necessary ammendments

In [6]:
# Check the crs of each layer
layer_crs = {'greenspace_crs': [greenspace.crs], 'lgd_crs': [lgd.crs], 'dz_crs': [dz.crs], 'sdz_crs': [sdz.crs]}
crs_table = pd.DataFrame(data=layer_crs)
crs_table

,greenspace_crs,lgd_crs,dz_crs,sdz_crs
0,EPSG:29902,EPSG:29902,EPSG:29902,EPSG:29902


In [7]:
ni_utm = ccrs.UTM(29) # create a Universal Transverse Mercator reference system to transform our data.

In [8]:
ccrs.CRS(greenspace.crs) # create a cartopy CRS representation of the CRS associated with the outline dataset


<Projected CRS: EPSG:29902>
Name: TM65 / Irish Grid
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Ireland - onshore.
- bounds: (-10.56, 51.39, -5.93, 55.43)
Coordinate Operation:
- name: Irish Grid
- method: Transverse Mercator
Datum: TM65
- Ellipsoid: Airy Modified 1849
- Prime Meridian: Greenwich

## Part 2: Greenspace Analysis

This next part of the notebook will demonstrate some analysis using the greenspace data and boundaries provided.

Using the data, we can find:
1. What areas have the highest/lowest coverage of greenspace: at SDZ level and DEA, and per 1000 people.
2. What areas (SDZ) are closest or within a certain distance of greenspaces
3. Bonus: How many parcels are available within each DEA that are available for greenspace? These must be over a certain size, be a natural landscape, within distance of roads, within distance of houses and not already within a greenspace. Which DEAs have the most of these?

So, lets begin...


### Calculating coverage

Before we can calculate what area of greenspace is found within each SDZ, LGD, DEA and so on. It would be useful to calculate the area of each polygon to a standard unit.
This is where our first fucntion will be introduced, one that calculates the area of each polygon, in Km2 and adds it on as a new column to the table.

In [9]:
def area_calc(layer):
    """Caluclates the area of each polygon in the layer and returns a total.
    
    Parameters 
    layer : input polygon layer
    name : name of the layer

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers
    """

    layer['Area_SQKM'] = layer['geometry'].area/1e6

    sum_area_SQKM = layer['Area_SQKM'].sum()
    print(f'The total area is {sum_area_SQKM:.2f} kilometers squared')

In [10]:
#test area_calc function on greenspace layer
area_calc(greenspace)

The total area is 875.94 kilometers squared


In [18]:
#test area_calc function column on greenspace layer
greenspace

,SourceID,GUID,Name,Source,Category,Type,PaidAccess,Area_Ha,Verified,ShowOnMap,ORNI_ID,DataAdded,SiteCreate,geometry,Area_SQKM
0,NaN,68d2df20-1512-4218-a759-1000732e93b0,Conservation Volunteers NI,LPS and Outscape,Amenity Greenspace,Community Garden,No,0.668234,Approximated,Yes,51,2022-09-21,Pre 2023,"POLYGON Z ((335088.596 367463.239 0, 334995.26...",0.006682
1,NaN,7b20f220-682e-4566-a4fe-2513cc65ed6e,Lough Shore Park,Antrim and Newtownabbey Borough Council,Parks and Gardens,Public Park,No,6.598284,Approximated,Yes,61,2022-09-21,Pre 2023,"POLYGON Z ((313511.357 386601.644 0, 313465.51...",0.065983
2,NaN,8a4b8a31-eba9-4e7a-b8d5-cea0724d848d,Ardmore Recreation Centre,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Playing Fields,No,2.969549,Approximated,Yes,67,2022-09-21,Pre 2023,"POLYGON Z ((289345.456 344248.549 0, 289352.83...",0.029695
3,NaN,36f2a040-649c-4eb5-9174-7b25c083f489,Clare Glen - Phase 3 - Link Footpath/River Wal...,"Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,3.396109,Approximated,Yes,69,2022-09-21,Pre 2023,"POLYGON Z ((303292.783 345625.097 0, 303291.38...",0.033961
4,NaN,2f586e46-b9ee-4d13-83b1-9b254fa3a698,"Folly Glen, Armagh","Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,7.849400,Approximated,Yes,70,2022-09-21,Pre 2023,"MULTIPOLYGON Z (((288701.465 343901.121 0, 288...",0.078494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1486,NaN,NaN,Connswater Community Greenway,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,7.872995,NaN,Yes,0,1899-12-30,Pre 2023,"MULTIPOLYGON Z (((338098.461 372640.049 0, 338...",0.078730
1487,NaN,NaN,Twinbrook Wildlife Park,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,1.161658,NaN,Yes,0,1899-12-30,NaN,"MULTIPOLYGON Z (((328479.994 368987.968 0, 328...",0.011617
1488,NaN,NaN,Drumcoo Playing Fields,Mid Ulster District Council,Amenity Greenspace,Playing Fields,NaN,3.022465,NaN,Yes,0,1899-12-30,Pre 2023,"POLYGON Z ((279812.935 363921.041 0, 279810.78...",0.030225
1489,NaN,NaN,Battery Harbour Park,Mid Ulster District Council,Amentiy Greenspace,Public Open Space,NaN,1.769502,NaN,Yes,0,1899-12-30,NaN,"POLYGON Z ((296502.851 377054.243 0, 296514.15...",0.017695


In [21]:
help(area_calc)

Help on function area_calc in module __main__:

area_calc(layer)
    Caluclates the area of each polygon in the layer and returns a total.

    Parameters
    layer : input polygon layer

    Returns : sum_area_SQKM
        Prints the total area of the polygons in squared kilometers



In [12]:
#add a area SQKM column to the remainder of the datasets and find their total area
area_calc(dz)
area_calc(sdz)
area_calc(dea)
area_calc(lgd)

The total area is 13628.31 kilometers squared
The total area is 13628.31 kilometers squared
The total area is 14315.28 kilometers squared
The total area is 14315.28 kilometers squared


Now we are ready to start analysing.

In [20]:
#Test!! Try using intersect function
intersect_greenspace = gpd.overlay(greenspace, dea, how='intersection')

C:\Users\danie\anaconda3\envs\project722\Lib\site-packages\geopandas\tools\overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 2 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [21]:
intersect_greenspace

,SourceID,GUID,Name,Source,Category,Type,PaidAccess,Area_Ha,Verified,ShowOnMap,...,R_Criminal,R_Drugs,R_Robbery,R_Shoplift,R_Vehicle_,R_Violence,Shape_Leng,Shape_Area,Area_SQKM_2,geometry
0,NaN,68d2df20-1512-4218-a759-1000732e93b0,Conservation Volunteers NI,LPS and Outscape,Amenity Greenspace,Community Garden,No,0.668234,Approximated,Yes,...,3.995410,3.145322,0.085009,2.465253,0.850087,11.773707,36427.709692,2.592510e+07,25.925099,"POLYGON Z ((334995.265 367429.786 0, 334980.36..."
1,NaN,7b20f220-682e-4566-a4fe-2513cc65ed6e,Lough Shore Park,Antrim and Newtownabbey Borough Council,Parks and Gardens,Public Park,No,6.598284,Approximated,Yes,...,14.109723,5.072680,0.255765,4.006991,1.236199,30.180315,24142.561884,2.350514e+07,23.505139,"POLYGON Z ((313465.518 386624.927 0, 313441.50..."
2,NaN,8a4b8a31-eba9-4e7a-b8d5-cea0724d848d,Ardmore Recreation Centre,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Playing Fields,No,2.969549,Approximated,Yes,...,7.304556,3.054060,0.031485,2.833664,1.542773,15.396241,115202.628186,2.772606e+08,277.260631,"POLYGON Z ((289352.835 344228.7 0, 289353.317 ..."
3,NaN,36f2a040-649c-4eb5-9174-7b25c083f489,Clare Glen - Phase 3 - Link Footpath/River Wal...,"Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,3.396109,Approximated,Yes,...,4.652272,1.550757,0.039763,0.835023,1.510994,9.423834,107279.963102,3.254195e+08,325.419542,"POLYGON Z ((303291.381 345617.806 0, 303291.80..."
4,NaN,2f586e46-b9ee-4d13-83b1-9b254fa3a698,"Folly Glen, Armagh","Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,7.849400,Approximated,Yes,...,7.304556,3.054060,0.031485,2.833664,1.542773,15.396241,115202.628186,2.772606e+08,277.260631,"MULTIPOLYGON Z (((288685.826 343903.039 0, 288..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,NaN,NaN,Connswater Community Greenway,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,7.872995,NaN,Yes,...,6.045214,1.752754,0.143082,3.541279,1.895836,12.161969,15267.660488,8.282405e+06,8.282405,"MULTIPOLYGON Z (((337767.884 372807.038 0, 337..."
1618,NaN,NaN,Twinbrook Wildlife Park,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,1.161658,NaN,Yes,...,9.659708,2.172700,0.293608,0.763381,1.673566,17.851376,19600.843226,1.060261e+07,10.602614,"MULTIPOLYGON Z (((328473.321 368975.896 0, 328..."
1619,NaN,NaN,Drumcoo Playing Fields,Mid Ulster District Council,Amenity Greenspace,Playing Fields,NaN,3.022465,NaN,Yes,...,7.085917,3.180610,0.161044,3.220871,2.174088,19.969402,60660.873508,9.410041e+07,94.100411,"POLYGON Z ((279810.78 363922.651 0, 279803.279..."
1620,NaN,NaN,Battery Harbour Park,Mid Ulster District Council,Amentiy Greenspace,Public Open Space,NaN,1.769502,NaN,Yes,...,2.718600,1.878306,0.148287,0.543720,1.334586,7.414364,128467.569899,3.241617e+08,324.161671,"POLYGON Z ((296514.158 377049.184 0, 296519.03..."


In [22]:
area_calc(intersect_greenspace)

The total area is 863.02 kilometers squared


In [67]:
intersect_greenspace

,SourceID,GUID,Name,Source,Category,Type,PaidAccess,Area_Ha,Verified,ShowOnMap,...,R_Drugs,R_Robbery,R_Shoplift,R_Vehicle_,R_Violence,Shape_Leng,Shape_Area,Area_SQKM_2,geometry,Area_SQKM
0,NaN,68d2df20-1512-4218-a759-1000732e93b0,Conservation Volunteers NI,LPS and Outscape,Amenity Greenspace,Community Garden,No,0.668234,Approximated,Yes,...,3.145322,0.085009,2.465253,0.850087,11.773707,36427.709692,2.592510e+07,25.925099,"POLYGON Z ((334995.265 367429.786 0, 334980.36...",0.006682
1,NaN,7b20f220-682e-4566-a4fe-2513cc65ed6e,Lough Shore Park,Antrim and Newtownabbey Borough Council,Parks and Gardens,Public Park,No,6.598284,Approximated,Yes,...,5.072680,0.255765,4.006991,1.236199,30.180315,24142.561884,2.350514e+07,23.505139,"POLYGON Z ((313465.518 386624.927 0, 313441.50...",0.065983
2,NaN,8a4b8a31-eba9-4e7a-b8d5-cea0724d848d,Ardmore Recreation Centre,"Armagh City, Banbridge and Craigavon Borough C...",Amenity Greenspace,Playing Fields,No,2.969549,Approximated,Yes,...,3.054060,0.031485,2.833664,1.542773,15.396241,115202.628186,2.772606e+08,277.260631,"POLYGON Z ((289352.835 344228.7 0, 289353.317 ...",0.029695
3,NaN,36f2a040-649c-4eb5-9174-7b25c083f489,Clare Glen - Phase 3 - Link Footpath/River Wal...,"Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,3.396109,Approximated,Yes,...,1.550757,0.039763,0.835023,1.510994,9.423834,107279.963102,3.254195e+08,325.419542,"POLYGON Z ((303291.381 345617.806 0, 303291.80...",0.033961
4,NaN,2f586e46-b9ee-4d13-83b1-9b254fa3a698,"Folly Glen, Armagh","Armagh City, Banbridge and Craigavon Borough C...",Woodland,Woodland,No,7.849400,Approximated,Yes,...,3.054060,0.031485,2.833664,1.542773,15.396241,115202.628186,2.772606e+08,277.260631,"MULTIPOLYGON Z (((288685.826 343903.039 0, 288...",0.078494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1617,NaN,NaN,Connswater Community Greenway,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,7.872995,NaN,Yes,...,1.752754,0.143082,3.541279,1.895836,12.161969,15267.660488,8.282405e+06,8.282405,"MULTIPOLYGON Z (((337767.884 372807.038 0, 337...",0.042585
1618,NaN,NaN,Twinbrook Wildlife Park,Belfast City Council,Amenity Greenspace,Public Open Space,NaN,1.161658,NaN,Yes,...,2.172700,0.293608,0.763381,1.673566,17.851376,19600.843226,1.060261e+07,10.602614,"MULTIPOLYGON Z (((328473.321 368975.896 0, 328...",0.011617
1619,NaN,NaN,Drumcoo Playing Fields,Mid Ulster District Council,Amenity Greenspace,Playing Fields,NaN,3.022465,NaN,Yes,...,3.180610,0.161044,3.220871,2.174088,19.969402,60660.873508,9.410041e+07,94.100411,"POLYGON Z ((279810.78 363922.651 0, 279803.279...",0.030225
1620,NaN,NaN,Battery Harbour Park,Mid Ulster District Council,Amentiy Greenspace,Public Open Space,NaN,1.769502,NaN,Yes,...,1.878306,0.148287,0.543720,1.334586,7.414364,128467.569899,3.241617e+08,324.161671,"POLYGON Z ((296514.158 377049.184 0, 296519.03...",0.017695


In [108]:
grouped=pd.DataFrame(
    intersect_greenspace.groupby(['DEA'])['Area_SQKM'].sum())
grouped

,Area_SQKM
DEA,
AIRPORT,1.588849
ANTRIM,1.661108
ARDS PENINSULA,5.609189
ARMAGH,3.313183
BALLYARNETT,1.440889
...,...
THREE MILE WATER,0.743206
TITANIC,0.636363
TORRENT,2.117588
